# Comparing Prediction Scenario Results

In [10]:
# Load Prediction Results
import pandas as pd
import polars as pl
#import pyarrow as pa
#import pyarrow.parquet as pq

future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'

filename1 = 'torch_predictions_current_All_COMID_sw.parquet'  
filename2 = 'torch_pred_scenario_half_Surp_COMID_sw.parquet'
filename3 = 'torch_pred_scenario_fut_RCP4.5G_COMID_sw.parquet'
filename4 = 'torch_pred_scenario_fut_RCP8.5G_COMID_sw.parquet'
filename5 = 'torch_pred_scenario_fut_RCP4.5H_COMID_sw.parquet'
filename6 = 'torch_pred_scenario_fut_RCP8.5H_COMID_sw.parquet'

filename7 = 'torch_pred_scenario_nue_COMID_sw.parquet'
filename8 = 'torch_pred_scenario_prod55_nue0_HUC12_sw.parquet'
filename9 = 'torch_pred_scenario_prod55_nue15_HUC12_sw.parquet'


# Read the file
preds_curr = pl.read_parquet(future_dir + filename1)
preds_half = pl.read_parquet(future_dir + filename2)

preds_RCP4_5G = pl.read_parquet(future_dir + filename3)
preds_RCP8_5G = pl.read_parquet(future_dir + filename4)
preds_RCP4_5H = pl.read_parquet(future_dir + filename5)
preds_RCP8_5H = pl.read_parquet(future_dir + filename6)

preds_nue = pl.read_parquet(future_dir + filename7)
preds_prod55_nue0 = pl.read_parquet(future_dir + filename8)
preds_prod55_nue15 = pl.read_parquet(future_dir + filename9)

# convert to pandas df
preds_curr = preds_curr.to_pandas()
preds_half = preds_half.to_pandas()
preds_RCP4_5G = preds_RCP4_5G.to_pandas()
preds_RCP8_5G = preds_RCP8_5G.to_pandas()
preds_RCP4_5H = preds_RCP4_5H.to_pandas()
preds_RCP8_5H = preds_RCP8_5H.to_pandas()
preds_nue = preds_nue.to_pandas()
preds_prod55_nue0 = preds_prod55_nue0.to_pandas()
preds_prod55_nue15 = preds_prod55_nue15.to_pandas()



In [ ]:
value1 = 100*len(preds_curr[preds_curr['Pred_Viol_Prob']>0.5])/len(preds_curr)
value2 = 100*len(preds_half[preds_half['Pred_Viol_Prob']>0.5])/len(preds_half)
value3 = 100*len(preds_nue[preds_nue['Pred_Viol_Prob']>0.5])/len(preds_nue)
value4 = 100*len(preds_prod55_nue0[preds_prod55_nue0['Pred_Viol_Prob']>0.5])/len(preds_prod55_nue0)
value5 = 100*len(preds_RCP4_5G[preds_RCP4_5G['Pred_Viol_Prob']>0.5])/len(preds_RCP4_5G)
value6 = 100*len(preds_RCP8_5G[preds_RCP8_5G['Pred_Viol_Prob']>0.5])/len(preds_RCP8_5G)
value7 = 100*len(preds_RCP4_5H[preds_RCP4_5H['Pred_Viol_Prob']>0.5])/len(preds_RCP4_5H)
value8 = 100*len(preds_RCP8_5H[preds_RCP8_5H['Pred_Viol_Prob']>0.5])/len(preds_RCP8_5H)

print(f"{value1:.2f}% of the Current predictions are above 0.5")
print(f"{value2:.2f}% of the Half Surplus predictions are above 0.5")
print(f"{value3:.2f}% of the NUE predictions are above 0.5")
print(f"{value4:.2f}% of the Production 55, NUE 15 predictions are above 0.5")
print(f"{value5:.2f}% of the RCP 4.5G predictions are above 0.5")
print(f"{value6:.2f}% of the RCP 8.5G predictions are above 0.5")
print(f"{value7:.2f}% of the RCP 4.5H predictions are above 0.5")
print(f"{value8:.2f}% of the RCP 8.5H predictions are above 0.5")

15.44% of the Currentpredictions are above 0.5
15.07% of the Half Surplus predictions are above 0.5
16.16% of the NUE predictions are above 0.5
21.75% of the Production 55, NUE 15 predictions are above 0.5
11.17% of the RCP 4.5G predictions are above 0.5
11.30% of the RCP 8.5G predictions are above 0.5
12.05% of the RCP 4.5H predictions are above 0.5
12.49% of the RCP 8.5H predictions are above 0.5


# Create Different Maps

In [16]:
# Merge

preds_diff = preds_curr.merge(preds_half, on="COMID", how="left")
#preds_diff.columns

preds_diff['Diff'] = preds_diff['Pred_Viol_Prob_y'] - preds_diff['Pred_Viol_Prob_x']
preds_diff['Diff'].mean(), preds_diff['Diff'].std(), preds_diff['Diff'].min(), preds_diff['Diff'].max()


(np.float32(-0.003896414),
 np.float32(0.35730222),
 np.float32(-1.0),
 np.float32(1.0))

# Import Shapefiles

In [17]:
# States boundary file
import geopandas as gpd
states_path = 'D:/ArcGIS/Boundaries/States/conterminous_states_usgs/Conterminous_States_usgs.shp'
states = gpd.read_file(states_path)

In [18]:
# Read in NHD Catchments Shapefile (2m 11s)
import fiona
# 1. Define the path to your .gdb folder
gdb_path = 'D:/ArcGIS/NHDplusV2/Seamless/NHDPlusNationalData/NHDPlusV21_National_Seamless.gdb'

# 2. List all feature classes (layers) inside the geodatabase
layers = fiona.listlayers(gdb_path)
print("Available layers:", layers)

# 3. Read a specific feature class into a GeoDataFrame
# (Replace 'your_layer_name' with one of the printed layers)
cats_s = gpd.read_file(gdb_path, layer="Catchment")
cats_s.shape

Available layers: ['Gage', 'BurnAddLine', 'BurnAddWaterbody', 'LandSea', 'Sink', 'Wall', 'Catchment', 'CatchmentSP', 'NHDArea', 'NHDWaterbody', 'NHDPlusComponentVersions', 'PlusARPointEvent', 'PlusFlow', 'PlusFlowAR', 'NHDFCode', 'DivFracMP', 'BurnLineEvent', 'NHDFlowline_Network', 'NHDFlowline_NonNetwork', 'HUC12', 'GeoNetwork_Junctions', 'N_1_Desc', 'N_1_EDesc', 'N_1_EStatus', 'N_1_ETopo', 'N_1_FloDir', 'N_1_JDesc', 'N_1_JStatus', 'N_1_JTopo', 'N_1_JTopo2', 'N_1_Props']


c:\Users\MPennino\anaconda3\envs\map_env\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP. Further messages of this type will be suppressed.
  return ogr_read(
c:\Users\MPennino\anaconda3\envs\map_env\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined. Further messages of this type will be suppressed.
  return ogr_read(


(2647454, 7)

In [ ]:
cats_s = cats_s.rename(columns={'FEATUREID': 'COMID'})
# Change projection (1 min 34s)
cats_meters = cats_s.to_crs("EPSG:5070")

# Merge with Shapefile

In [ ]:
# Merge Prediction results with Shapefile
#type(cats_s), cats_s.columns
preds_s = cats_meters.merge(preds_diff, on="COMID", how="left")
preds_s.shape, preds_s.columns, type(preds_s)



# Generate Raster

In [ ]:
# Run this iteratively as separate chunks (~6m20s for 1000m, 6m25s for 3000m, 38m53s for 300m)
import geopandas as gpd
from geocube.api.core import make_geocube
import xarray as xr

# Fix the Xarray future warning globally
#xr.set_options(use_new_combine_kwarg_defaults=True)

chunk_size = 50000
total_rows = len(preds_s)
cubes = []

for start_idx in range(0, total_rows, chunk_size):
    end_idx = min(start_idx + chunk_size, total_rows)

    gdf_chunk = preds_s.iloc[start_idx:end_idx]

    cube_chunk = make_geocube(
        vector_data=gdf_chunk,
        measurements=["Diff"],  # Column containing pixel values
        resolution=(-1000, 1000)                  # Pixel size (Y, X) - Y is usually negative
        #like=master_grid,  # aligns subsequent chunks to the first grid
    )

    cubes.append(cube_chunk)